## EVALUACIÓN EN MIXED TEST (15 VALDO Y 50 REAL ADNI)

Métricas de detección de los distintos modelos evaluados sobre el conjunto de test mixto (presente en imagesTs y labelsTs de D205, pero es el conjunto de test usado para todos).

Se incluyen en la memoria del TFM:
- D201: 200 synth sCMB ADNI gaussian
- D211: igual que D201 pero sin sujetos sanos
- D203: 200 synth sCMB LesionGAN
- D202: 200 real ADNI
- D800: 57 real VALDO
- D205: 57 real VALDO + 200 synth ADNI gaussian
- D207: 57 real VALDO + 100 real ADNI + 100 synth ADNI gaussian
- D208: 57 real VALDO + 200 real ADNI


### D201 

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_201_newspacing"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs"  
#OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS_MIXTEST/201_analysis_results"
OUTPUT_DIR = r"C:\Users\marta\Downloads\PREDICTS_ON_VALDO\predicts_from_201_newspacing"


os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50 105  13 326      0.8898     0.2436    0.3825      0.2682     0.1557    0.1846
  VALDO        15   2   3  49      0.4000     0.0392    0.0714      0.1000     0.1000    0.1000

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50 105  13 326      0.8898     0.2436    0.3825      0.2682     0.1557    0.1846
  ALFA         6   1   0   7      1.0000     0.1250    0.2222      0.1667     0.1667    0.1667
   RSS         7   1   3  41      0.2500     0.0238    0.0435      0.0714     0.0714    0.0714
 SABRE         2   0   0   1      0.0000     0.0000    0.0000      0.0000     0.0000    0.0000

==================

### D211

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_211_PROB"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs"  
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS_MIXTEST/211_analysis_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50 111  15 320       0.881     0.2575    0.3986      0.3325     0.2028    0.2316
  VALDO        15   2   3  49       0.400     0.0392    0.0714      0.1333     0.0889    0.1000

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50 111  15 320       0.881     0.2575    0.3986      0.3325     0.2028    0.2316
  ALFA         6   2   0   6       1.000     0.2500    0.4000      0.3333     0.2222    0.2500
   RSS         7   0   3  42       0.000     0.0000    0.0000      0.0000     0.0000    0.0000
 SABRE         2   0   0   1       0.000     0.0000    0.0000      0.0000     0.0000    0.0000

==================

### D203

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_203_PROB"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs"  
#OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS_MIXTEST/203_analysis_results"
OUTPUT_DIR = r"C:\Users\marta\Downloads\PREDICTS_ON_VALDO\predicts_from_203_PROB"


os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50 113  25 318      0.8188     0.2622    0.3972      0.4088     0.2928    0.3149
  VALDO        15   6   4  45      0.6000     0.1176    0.1967      0.3111     0.2470    0.2494

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50 113  25 318      0.8188     0.2622    0.3972      0.4088     0.2928    0.3149
  ALFA         6   4   1   4      0.8000     0.5000    0.6154      0.4444     0.4444    0.4444
   RSS         7   1   3  41      0.2500     0.0238    0.0435      0.1429     0.0055    0.0106
 SABRE         2   1   0   0      1.0000     1.0000    1.0000      0.5000     0.5000    0.5000

==================

### D202

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_202_PROB"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs"  
#OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS_MIXTEST/202_analysis_results"
OUTPUT_DIR = r"C:\Users\marta\Downloads\PREDICTS_ON_VALDO\predicts_from_202_PROB"


os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50 287 128 144      0.6916     0.6659    0.6785      0.6133     0.5514    0.5485
  VALDO        15  21  20  30      0.5122     0.4118    0.4565      0.4100     0.4070    0.3930

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50 287 128 144      0.6916     0.6659    0.6785      0.6133     0.5514    0.5485
  ALFA         6   5   3   3      0.6250     0.6250    0.6250      0.5000     0.5000    0.5000
   RSS         7  15  17  27      0.4688     0.3571    0.4054      0.3071     0.3007    0.2708
 SABRE         2   1   0   0      1.0000     1.0000    1.0000      0.5000     0.5000    0.5000

==================

### D800

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_800_3d_fullres_smallpatch"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs" 
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS_MIXTEST/800_analysis_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50 150 103 281      0.5929     0.3480    0.4386      0.3393     0.3257    0.2940
  VALDO        15  37  21  14      0.6379     0.7255    0.6789      0.4569     0.5099    0.4609

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50 150 103 281      0.5929     0.3480    0.4386      0.3393     0.3257    0.2940
  ALFA         6   4   4   4      0.5000     0.5000    0.5000      0.4444     0.4444    0.4444
   RSS         7  32  14  10      0.6957     0.7619    0.7273      0.5505     0.5689    0.5353
 SABRE         2   1   3   0      0.2500     1.0000    0.4000      0.1667     0.5000    0.2500

==================

### D205

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_205_newspacing"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs"  
#OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS_MIXTEST/205_analysis_results"
OUTPUT_DIR = r"C:\Users\marta\Downloads\PREDICTS_ON_VALDO\predicts_from_205_newspacing"


os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50 127  18 304      0.8759     0.2947    0.4410      0.3684     0.2427    0.2673
  VALDO        15  17   5  34      0.7727     0.3333    0.4658      0.4056     0.3428    0.3640

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50 127  18 304      0.8759     0.2947    0.4410      0.3684     0.2427    0.2673
  ALFA         6   5   0   3      1.0000     0.6250    0.7692      0.6667     0.6111    0.6333
   RSS         7  12   5  30      0.7059     0.2857    0.4068      0.2976     0.2108    0.2371
 SABRE         2   0   0   1      0.0000     0.0000    0.0000      0.0000     0.0000    0.0000

==================

### D207

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_207_PROB_corrected"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs"  
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS_MIXTEST/207_analysis_results_corrected"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50 243  64 188      0.7915     0.5638    0.6585      0.6235      0.498    0.5114
  VALDO        15  17   4  34      0.8095     0.3333    0.4722      0.4259      0.366    0.3816

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50 243  64 188      0.7915     0.5638    0.6585      0.6235     0.4980    0.5114
  ALFA         6   7   1   1      0.8750     0.8750    0.8750      0.8333     0.8333    0.8333
   RSS         7  10   3  32      0.7692     0.2381    0.3636      0.1984     0.0699    0.1034
 SABRE         2   0   0   1      0.0000     0.0000    0.0000      0.0000     0.0000    0.0000

==================

### D208

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_208_PROB_corrected"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs"  
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS_MIXTEST/208_analysis_results_corrected"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50 264 109 167      0.7078     0.6125    0.6567      0.5036     0.5624    0.4955
  VALDO        15  29  11  22      0.7250     0.5686    0.6374      0.5852     0.5960    0.5832

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50 264 109 167      0.7078     0.6125    0.6567      0.5036     0.5624    0.4955
  ALFA         6   8   0   0      1.0000     1.0000    1.0000      1.0000     1.0000    1.0000
   RSS         7  20  11  22      0.6452     0.4762    0.5479      0.2540     0.2772    0.2498
 SABRE         2   1   0   0      1.0000     1.0000    1.0000      0.5000     0.5000    0.5000

==================

### Folds 1-4 del D208 (anterior era el Fold0):

Fold 1

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_208_sinPROB_fold1"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs"  
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS_MIXTEST/208_fold1_analysis_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50 260  70 171      0.7879     0.6032    0.6833      0.6572     0.5383    0.5505
  VALDO        15  13   6  38      0.6842     0.2549    0.3714      0.4267     0.4163    0.3886

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50 260  70 171      0.7879     0.6032    0.6833      0.6572     0.5383    0.5505
  ALFA         6   7   3   1      0.7000     0.8750    0.7778      0.6833     0.8333    0.7361
   RSS         7   6   3  36      0.6667     0.1429    0.2353      0.3286     0.1778    0.2017
 SABRE         2   0   0   1      0.0000     0.0000    0.0000      0.0000     0.0000    0.0000

==================

Fold 2

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_208_sinPROB_fold2"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs"  
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS_MIXTEST/208_fold2_analysis_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50 273  97 158      0.7378     0.6334    0.6816      0.6044     0.5154    0.5013
  VALDO        15  22   9  29      0.7097     0.4314    0.5366      0.5468     0.5070    0.5201

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50 273  97 158      0.7378     0.6334    0.6816      0.6044     0.5154    0.5013
  ALFA         6   6   2   2      0.7500     0.7500    0.7500      0.6667     0.6667    0.6667
   RSS         7  15   7  27      0.6818     0.3571    0.4688      0.4575     0.3721    0.4002
 SABRE         2   1   0   0      1.0000     1.0000    1.0000      0.5000     0.5000    0.5000

==================

Fold 3

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_208_sinPROB_fold3"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs"  
#OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS_MIXTEST/208_analysis_results_corrected/fold3"
OUTPUT_DIR = r"C:\Users\marta\Downloads\PREDICTS_ON_VALDO\predicts_from_208_sinPROB_fold3"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("D208: fold3")
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)
D208: fold3

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50 257  74 174      0.7764     0.5963    0.6745      0.6546     0.5328    0.5420
  VALDO        15  20   6  31      0.7692     0.3922    0.5195      0.5444     0.4984    0.5113

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50 257  74 174      0.7764     0.5963    0.6745      0.6546     0.5328    0.5420
  ALFA         6   7   1   1      0.8750     0.8750    0.8750      0.8333     0.8333    0.8333
   RSS         7  12   5  30      0.7059     0.2857    0.4068      0.3095     0.2108    0.2385
 SABRE         2   1   0   0      1.0000     1.0000    1.0000      0.5000     0.5000    0.5000

======

Fold 4

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_208_sinPROB_fold4"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs"  
#OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS_MIXTEST/208_analysis_results_corrected/fold4"
OUTPUT_DIR = r"C:\Users\marta\Downloads\PREDICTS_ON_VALDO\predicts_from_208_sinPROB_fold4"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("D208: fold4")
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)
D208: fold4

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50 247  60 184      0.8046     0.5731    0.6694      0.6462     0.5352    0.5350
  VALDO        15  25   7  26      0.7812     0.4902    0.6024      0.6505     0.6096    0.6122

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50 247  60 184      0.8046     0.5731    0.6694      0.6462     0.5352    0.5350
  ALFA         6   7   1   1      0.8750     0.8750    0.8750      0.7500     0.8333    0.7778
   RSS         7  17   6  25      0.7391     0.4048    0.5231      0.6082     0.4491    0.5024
 SABRE         2   1   0   0      1.0000     1.0000    1.0000      0.5000     0.5000    0.5000

======

Ensemble

In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_208_sinPROB_ensemble"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs"  
#OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS_MIXTEST/208_analysis_results_corrected/ensemble"
OUTPUT_DIR = r"C:\Users\marta\Downloads\PREDICTS_ON_VALDO\predicts_from_208_sinPROB_ensemble"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("D208: ensemble")
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)
D208: ensemble

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50 251  53 180      0.8257     0.5824    0.6830      0.6728     0.5170    0.5419
  VALDO        15  20   8  31      0.7143     0.3922    0.5063      0.4731     0.4343    0.4465

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50 251  53 180      0.8257     0.5824    0.6830      0.6728     0.5170    0.5419
  ALFA         6   6   2   2      0.7500     0.7500    0.7500      0.6667     0.6667    0.6667
   RSS         7  13   6  29      0.6842     0.3095    0.4262      0.2995     0.2163    0.2425
 SABRE         2   1   0   0      1.0000     1.0000    1.0000      0.5000     0.5000    0.5000

===

### Otras pruebas no incluidas en la memoria:

Ninguna obtuvo resultados significativamente mejores, así que solo quedan recogidas aquí como trials paralelos

### D201 3d fullres_smallpatch 250epochs (only sCMBs ADNI) + ft 800 3d fullres smallpatch 50epochs LR0.001

In [1]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_201ft800_3d_fullres_smallpatch"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs"  # Asegúrate de cambiar XXXX
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS/201ft800_analysis_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50  84   7 347      0.9231     0.1949    0.3218      0.3583     0.1818    0.2219
  VALDO        15  11   1  40      0.9167     0.2157    0.3492      0.3333     0.2795    0.2882

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50  84   7 347      0.9231     0.1949    0.3218      0.3583     0.1818    0.2219
  ALFA         6   4   1   4      0.8000     0.5000    0.6154      0.3333     0.3333    0.3333
   RSS         7   6   0  36      1.0000     0.1429    0.2500      0.2857     0.1703    0.1889
 SABRE         2   1   0   0      1.0000     1.0000    1.0000      0.5000     0.5000    0.5000

==================

### D201 Tversky a03 3d fullres smallpatch 100epochs NEW SPACING (MODELO SENSIBLE?)

In [1]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_201_tversky_PROB"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs"  
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS_MIXTEST/201_tversky_analysis_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50  50   0 381      1.0000     0.1160    0.2079      1.0000     0.5912    0.6784
  VALDO        15  13   2  38      0.8667     0.2549    0.3939      0.8667     0.6642    0.6938

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50  50   0 381      1.0000     0.1160    0.2079      1.0000     0.5912    0.6784
  ALFA         6   6   0   2      1.0000     0.7500    0.8571      1.0000     0.8889    0.9167
   RSS         7   6   1  36      0.8571     0.1429    0.2449      0.8571     0.5185    0.5582
 SABRE         2   1   1   0      0.5000     1.0000    0.6667      0.5000     0.5000    0.5000

==================

### D204 (entrenado con 100 de D201 y 100 de D203)

In [1]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
from skimage.measure import label

# ========================= CONFIGURACIÓN DE RUTAS =========================
PRED_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/predicts_from_204_PROB"
GT_DIR = "/media/PORT-DISK/Practicas/nnUNet_raw_ADNI/Dataset205_MixCMB/labelsTs"  
OUTPUT_DIR = "/media/PORT-DISK/Practicas/MicroBleeds_Generation/PREDICTS_ON_VALDO/ANALYSIS_RESULTS_MIXTEST/204_analysis_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================= FUNCIONES DE APOYO =========================
def get_metadata(fname):
    """Identifica el dataset y la cohorte basándose en el nombre del archivo."""
    if fname.startswith("sub-1"):
        return "VALDO", "SABRE"
    elif fname.startswith("sub-2"):
        return "VALDO", "RSS"
    elif fname.startswith("sub-3"):
        return "VALDO", "ALFA"
    elif fname.startswith("I"):
        return "ADNI", "ADNI"
    else:
        return "UNKNOWN", "UNKNOWN"

def calculate_metrics(tp, fp, fn):
    """Calcula precision, recall y f1 de forma segura."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ========================= CARGA DE ARCHIVOS =========================
pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith(".nii.gz")])
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith(".nii.gz")])
common_files = sorted(list(set(pred_files) & set(gt_files)))

print(f"Total archivos encontrados: {len(common_files)} (VALDO + ADNI)")

results_list = []

# ========================= BUCLE DE PROCESAMIENTO =========================
for fname in common_files:
    dataset, cohort = get_metadata(fname)
    
    pred_path = os.path.join(PRED_DIR, fname)
    gt_path = os.path.join(GT_DIR, fname)

    try:
        pred_img = nib.load(pred_path).get_fdata()
        gt_img = nib.load(gt_path).get_fdata()
    except Exception as e:
        print(f"Error cargando {fname}: {e}")
        continue

    # Binarización y etiquetado de componentes conectadas (blobs)
    labeled_pred, num_pred = label(pred_img > 0, return_num=True)
    labeled_gt, num_gt = label(gt_img > 0, return_num=True)

    tp_count = 0
    used_pred_blobs = set()

    # Matching de blobs: Si un blob de GT toca algún blob de Pred
    for gt_id in range(1, num_gt + 1):
        gt_blob_mask = (labeled_gt == gt_id)
        overlapping_pred_ids = np.unique(labeled_pred[gt_blob_mask])
        overlapping_pred_ids = overlapping_pred_ids[overlapping_pred_ids > 0]

        assigned = None
        for pid in overlapping_pred_ids:
            if pid not in used_pred_blobs:
                assigned = pid
                break
        
        if assigned is not None:
            tp_count += 1
            used_pred_blobs.add(assigned)

    fp_count = num_pred - len(used_pred_blobs)
    fn_count = num_gt - tp_count
    
    prec, rec, f1 = calculate_metrics(tp_count, fp_count, fn_count)

    results_list.append({
        'ID': fname,
        'Dataset': dataset,
        'Cohort': cohort,
        'TP': tp_count,
        'FP': fp_count,
        'FN': fn_count,
        'GT_blobs': num_gt,
        'Pred_blobs': num_pred,
        'Precision': prec,
        'Recall': rec,
        'F1': f1
    })

# ========================= AGREGACIÓN Y ESTADÍSTICAS =========================
df_res = pd.DataFrame(results_list)

def aggregate_results(df, group_col):
    agg_list = []
    for name, group in df.groupby(group_col):
        # Micro-métricas (suma total)
        total_tp = group['TP'].sum()
        total_fp = group['FP'].sum()
        total_fn = group['FN'].sum()
        mi_prec, mi_rec, mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)
        
        # Macro-métricas (media de los sujetos)
        ma_prec = group['Precision'].mean()
        ma_rec = group['Recall'].mean()
        ma_f1 = group['F1'].mean()
        
        agg_list.append({
            group_col: name,
            'Subjects': len(group),
            'TP': total_tp, 'FP': total_fp, 'FN': total_fn,
            'Prec_Micro': round(mi_prec, 4), 'Rec_Micro': round(mi_rec, 4), 'F1_Micro': round(mi_f1, 4),
            'Prec_Macro': round(ma_prec, 4), 'Rec_Macro': round(ma_rec, 4), 'F1_Macro': round(ma_f1, 4)
        })
    return pd.DataFrame(agg_list)

# Generar tablas
df_cohort = aggregate_results(df_res, 'Cohort')
df_dataset = aggregate_results(df_res, 'Dataset')

# Resultados Globales (Micro y Macro de todo el conjunto test)
total_tp = df_res['TP'].sum()
total_fp = df_res['FP'].sum()
total_fn = df_res['FN'].sum()
g_mi_prec, g_mi_rec, g_mi_f1 = calculate_metrics(total_tp, total_fp, total_fn)

global_results = pd.DataFrame([{
    'Level': 'GLOBAL',
    'Prec_Micro': round(g_mi_prec, 4), 'Rec_Micro': round(g_mi_rec, 4), 'F1_Micro': round(g_mi_f1, 4),
    'Prec_Macro': round(df_res['Precision'].mean(), 4), 
    'Rec_Macro': round(df_res['Recall'].mean(), 4), 
    'F1_Macro': round(df_res['F1'].mean(), 4)
}])

# ========================= OUTPUT FINAL =========================
print("\n" + "="*30 + " RESULTADOS POR DATASET " + "="*30)
print(df_dataset.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS POR COHORTE " + "="*30)
print(df_cohort.to_string(index=False))

print("\n" + "="*30 + " RESULTADOS GLOBALES " + "="*30)
print(global_results.to_string(index=False))

# Guardar a CSV
df_res.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_cohort.to_csv(os.path.join(OUTPUT_DIR, "cohort_summary.csv"), index=False)

Total archivos encontrados: 65 (VALDO + ADNI)

============================== RESULTADOS POR DATASET ==============================
Dataset  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
   ADNI        50 142  29 289      0.8304     0.3295    0.4718      0.4446     0.3877    0.3792
  VALDO        15   5   5  46      0.5000     0.0980    0.1639      0.3000     0.2248    0.2160

============================== RESULTADOS POR COHORTE ==============================
Cohort  Subjects  TP  FP  FN  Prec_Micro  Rec_Micro  F1_Micro  Prec_Macro  Rec_Macro  F1_Macro
  ADNI        50 142  29 289      0.8304     0.3295    0.4718      0.4446     0.3877    0.3792
  ALFA         6   4   2   4      0.6667     0.5000    0.5714      0.5833     0.5556    0.5278
   RSS         7   1   3  41      0.2500     0.0238    0.0435      0.1429     0.0055    0.0106
 SABRE         2   0   0   1      0.0000     0.0000    0.0000      0.0000     0.0000    0.0000

==================